## Part 1: Setting up a basic semantic search system

In [ ]:
from openai import OpenAI
from datetime import datetime, timezone
import hashlib
import os
import time
from sentence_transformers import CrossEncoder
from tqdm import tqdm
import numpy as np
from torch import nn
import logging
from pinecone import Pinecone, ServerlessSpec

In [ ]:
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [ ]:
pinecone_key = os.environ.get("PINECONE_API_KEY")
if not pinecone_key or not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("Set PINECONE_API_KEY and OPENAI_API_KEY before running.")
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [ ]:
INDEX_NAME = "semantic-search-test"
NAMESPACE = "default"
ENGINE = "text-embedding-3-large"  # has vector size 3072

In [ ]:
pc = Pinecone(api_key=pinecone_key)

In [ ]:
# helper functions to get lists of embeddings from the OpenAI API
def get_embeddings(texts, engine=ENGINE):
    response = client.embeddings.create(input=texts, model=engine)

    return [d.embedding for d in list(response.data)]

In [ ]:
def get_embedding(text, engine=ENGINE):
    return get_embeddings([text], engine)[0]

In [ ]:
print(len(get_embedding("hi")), len(get_embeddings(["hi", "hello"])))  # (3072, 2)

In [ ]:
if INDEX_NAME not in pc.list_indexes().names():
    print(f"Creating index {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,  # The name of the index
        dimension=3072,  # The dimensionality of the vectors for our OpenAI embedder
        metric="cosine",  # The similarity metric to use when searching the index
        spec=ServerlessSpec(cloud="aws", region="us-west-2"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

In [ ]:
# Store the index as a variable
index = pc.Index(name=INDEX_NAME)
print(index)

In [ ]:
print(index.describe_index_stats())

In [ ]:
def my_hash(s):
    # Return the MD5 hash of the input string as a hexadecimal string
    return hashlib.md5(s.encode()).hexdigest()

In [ ]:
print(my_hash("I love to hash it"))  # ae76cc4dfd345ecaeea9b8ba0d5c3437

In [ ]:
def prepare_for_pinecone(texts, engine=ENGINE):
    # Get the current UTC date and time
    now = datetime.now(
        timezone.utc
    ).isoformat()  # Pinecone metadata must be JSON-serializable

    # Generate vector embeddings for each string in the input list, using the specified engine
    embeddings = get_embeddings(texts, engine=engine)

    # Create tuples of (hash, embedding, metadata) for each input string and its corresponding vector embedding
    # The my_hash() function is used to generate a unique hash for each string, and the datetime.utcnow() function is used to generate the current UTC date and time
    return [
        (
            my_hash(
                text
            ),  # A unique ID for each string, generated using the my_hash() function
            embedding,  # The vector embedding of the string
            dict(
                text=text, date_uploaded=now
            ),  # A dictionary of metadata, including the original text and the date it was uploaded
        )
        for text, embedding in zip(
            texts, embeddings
        )  # Iterate over each input string and its corresponding vector embedding
    ]

In [ ]:
texts = ["hi"]

In [ ]:
_id, embedding, metadata = prepare_for_pinecone(texts)[0]

In [ ]:
print("ID: ", _id, "\nLEN: ", len(embedding), "\nMETA:", metadata)

In [ ]:
def upload_texts_to_pinecone(
    texts, namespace=NAMESPACE, batch_size=None, show_progress_bar=False
):
    # Call the prepare_for_pinecone function to prepare the input texts for indexing
    total_upserted = 0
    if not batch_size:
        batch_size = len(texts)

    _range = range(0, len(texts), batch_size)
    for i in tqdm(_range) if show_progress_bar else _range:
        batch = texts[i : i + batch_size]
        prepared_texts = prepare_for_pinecone(batch)

        # Use the upsert() method of the index object to upload the prepared texts to Pinecone
        total_upserted += index.upsert(vectors=prepared_texts, namespace=namespace)[
            "upserted_count"
        ]

    return total_upserted

In [ ]:
# Call the upload_texts_to_pinecone() function with the input texts
upload_texts_to_pinecone(texts)

In [ ]:
def query_from_pinecone(query, top_k=3, include_metadata=True):
    # get embedding from THE SAME embedder as the documents
    query_embedding = get_embedding(query, engine=ENGINE)

    return index.query(
        vector=query_embedding,
        top_k=top_k,
        namespace=NAMESPACE,
        include_metadata=include_metadata,  # gets the metadata (dates, text, etc)
    ).get("matches")

In [ ]:
def delete_texts_from_pinecone(texts, namespace=NAMESPACE):
    # Compute the hash (id) for each text
    hashes = [hashlib.md5(text.encode()).hexdigest() for text in texts]

    # The ids parameter is used to specify the list of IDs (hashes) to delete
    return index.delete(ids=hashes, namespace=namespace)

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("google/xtreme", "MLQA.en.en")

In [ ]:
# rename test -> train and val -> test (as we will use it in later in this chapter)
dataset["train"] = dataset["test"]
dataset["test"] = dataset["validation"]
del dataset["validation"]

In [ ]:
print(dataset)

In [ ]:
print(dataset["train"][0], dataset["train"][1])

In [ ]:
unique_passages = list(set(dataset["test"]["context"]))
for idx in tqdm(range(0, len(unique_passages), 32)):
    passages = unique_passages[idx : idx + 32]
    upload_texts_to_pinecone(passages)

In [ ]:
print(len(unique_passages))  # 978

In [ ]:
print(index.describe_index_stats())

In [ ]:
print(dataset["test"][0])

In [ ]:
print(query_from_pinecone("Does an infection for Sandflies go away over time?"))

## Part 2: Making results more relevant with a cross-encoder

In [ ]:
def get_results_from_pinecone(
    query, top_k=3, re_rank_model=None, verbose=True, correct_hash=None
):
    results_from_pinecone = query_from_pinecone(query, top_k=top_k)
    if not results_from_pinecone:
        return []
    if verbose:
        print("Query:", query)
    final_results = []

    retrieved_correct_position, reranked_correct_position = None, None
    for idx, result_from_pinecone in enumerate(results_from_pinecone):
        if correct_hash and result_from_pinecone["id"] == correct_hash:
            retrieved_correct_position = idx

    if re_rank_model is not None:
        if verbose:
            print("Document ID (Hash)\t\tRetrieval Score\tCE Score\tText")

        sentence_combinations = [
            [query, result_from_pinecone["metadata"]["text"]]
            for result_from_pinecone in results_from_pinecone
        ]

        # Compute the similarity scores for these combinations
        similarity_scores = re_rank_model.predict(
            sentence_combinations, activation_fct=nn.Sigmoid()
        )

        # Sort the scores in decreasing order
        sim_scores_argsort = list(reversed(np.argsort(similarity_scores)))

        # Print the scores
        for idx in sim_scores_argsort:
            result_from_pinecone = results_from_pinecone[idx]
            if correct_hash and result_from_pinecone["id"] == correct_hash:
                reranked_correct_position = idx
            final_results.append(
                {
                    "score": similarity_scores[idx],
                    "id": result_from_pinecone["id"],
                    "metadata": result_from_pinecone["metadata"],
                }
            )
            if verbose:
                print(
                    f"{result_from_pinecone['id']}\t{result_from_pinecone['score']:.2f}\t{similarity_scores[idx]:.2f}\t{result_from_pinecone['metadata']['text'][:50]}"
                )

        return {
            "final_results": final_results,
            "retrieved_correct_position": retrieved_correct_position,
            "reranked_correct_position": reranked_correct_position,
        }

    if verbose:
        print("Document ID (Hash)\t\tRetrieval Score\tText")
    for result_from_pinecone in results_from_pinecone:
        final_results.append(result_from_pinecone)
        if verbose:
            print(
                f"{result_from_pinecone['id']}\t{result_from_pinecone['score']:.2f}\t{result_from_pinecone['metadata']['text'][:50]}"
            )

    return {
        "final_results": final_results,
        "retrieved_correct_position": retrieved_correct_position,
        "reranked_correct_position": reranked_correct_position,
    }

In [ ]:
# Pre-trained cross encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2", num_labels=1)

In [ ]:
q_to_hash = {data["question"]: my_hash(data["context"]) for data in dataset["test"]}

In [ ]:
unique_inputs = list(set(dataset["test"]["question"]))
print(len(unique_inputs))  # 1148

In [ ]:
query = unique_inputs[0]
print(query)

In [ ]:
for t in dataset["test"]:
    if t["question"] == query:
        print(t["context"])

In [ ]:
query_result = get_results_from_pinecone(
    query,
    top_k=2,  # grab 2 results
    re_rank_model=cross_encoder,
    correct_hash=q_to_hash[query],
    verbose=False,
)

In [ ]:
print(
    query_result["retrieved_correct_position"],
    query_result["reranked_correct_position"],
)  # (None, None)

In [ ]:
print(query_result)  # the right context isn't there!

In [ ]:
query_result = get_results_from_pinecone(
    query,
    top_k=10,  # grab 10 results
    re_rank_model=cross_encoder,
    correct_hash=q_to_hash[query],
    verbose=False,
)

In [ ]:
print(
    query_result["retrieved_correct_position"],
    query_result["reranked_correct_position"],
)  # (2, 2)

In [ ]:
print(query_result)  # the 2nd position has the right context

In [ ]:
test_sample = dataset["test"]

In [ ]:
TOP_K = 50

In [ ]:
logger.setLevel(logging.CRITICAL)

In [ ]:
predictions = []

In [ ]:
for question in tqdm(test_sample["question"]):
    r = get_results_from_pinecone(
        question,
        top_k=TOP_K,
        re_rank_model=cross_encoder,
        correct_hash=q_to_hash[question],
        verbose=False,
    )

    predictions.append(r)
    if len(predictions) % 100 == 0:
        retrieved_accuracy = sum(
            [_["retrieved_correct_position"] == 0 for _ in predictions]
        ) / len(predictions)
        re_ranked_accuracy = sum(
            [_["reranked_correct_position"] == 0 for _ in predictions]
        ) / len(predictions)

        print(f"Accuracy without re-ranking: {retrieved_accuracy}")
        print(f"Accuracy with re-ranking: {re_ranked_accuracy}")

In [ ]:
retrieved_accuracy = sum(
    [_["retrieved_correct_position"] == 0 for _ in predictions]
) / len(predictions)
re_ranked_accuracy = sum(
    [_["reranked_correct_position"] == 0 for _ in predictions]
) / len(predictions)

In [ ]:
print(f"Accuracy without re-ranking: {retrieved_accuracy}")  # 0.7543554006968641
print(f"Accuracy with re-ranking: {re_ranked_accuracy}")  # 0.8327526132404182

In [ ]:
import pandas as pd

In [ ]:
predictions_df = pd.DataFrame(predictions)
print(predictions_df.head())

In [ ]:
print(
    predictions_df[["retrieved_correct_position", "reranked_correct_position"]].mean()
)  # lower is better

In [ ]:
# do recall @ 1, 3, 5, 10
for k in (1, 3, 5, 10, 25, 50):
    embedding_only_recall = predictions_df[
        predictions_df["retrieved_correct_position"] < k
    ].shape[0]
    reranked_recall = predictions_df[
        predictions_df["reranked_correct_position"] < k
    ].shape[0]
    print(k, embedding_only_recall, reranked_recall)

## OPEN SOURCE ALTERNATIVE TO EMBEDDING

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
# load up our open source embedding model
bi_encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [ ]:
# Encode query and documents
docs = dataset["test"]["context"]
doc_emb = bi_encoder.encode(docs, batch_size=32, show_progress_bar=True)

In [ ]:
print(doc_emb.shape)  # (1148, 768)

In [ ]:
from sentence_transformers.util import semantic_search

In [ ]:
# Function to find most similar document
def find_most_similar(text, embeddings, documents, k=3):
    query_embedding = bi_encoder.encode([text], show_progress_bar=False)
    similarities = semantic_search(query_embedding, embeddings, top_k=k)
    return [
        (documents[sim["corpus_id"]], sim["score"], sim["corpus_id"])
        for sim in similarities[0]
    ]

In [ ]:
from random import sample

In [ ]:
query = sample(dataset["test"]["question"], 1)[0]
print(query)

In [ ]:
def eval_ranking_open_source(query, top_k=3, re_rank_model=None):
    ans = {"retrieved_correct_position": None}
    correct_hash = q_to_hash[query]
    results = find_most_similar(query, doc_emb, docs, k=top_k)
    for idx, (passage, score, doc_idx) in enumerate(results):
        if correct_hash == my_hash(passage):
            ans["retrieved_correct_position"] = idx
    if re_rank_model is not None:
        ans["reranked_correct_position"] = None
        sentence_combinations = [(query, r[0]) for r in results]

        # Compute the similarity scores for these combinations
        similarity_scores = re_rank_model.predict(
            sentence_combinations, activation_fct=nn.Sigmoid()
        )

        # Sort the scores in decreasing order
        sim_scores_argsort = list(reversed(np.argsort(similarity_scores)))
        for i, idx in enumerate(sim_scores_argsort):
            r = results[idx]
            if correct_hash and my_hash(r[0]) == correct_hash:
                ans["reranked_correct_position"] = i

    return ans

In [ ]:
print(eval_ranking_open_source(query, top_k=TOP_K, re_rank_model=cross_encoder))

In [ ]:
logger.setLevel(logging.CRITICAL)
os_predictions = []

In [ ]:
for i, question in tqdm(enumerate(test_sample), total=len(test_sample)):
    os_predictions.append(
        eval_ranking_open_source(
            question["question"], top_k=TOP_K, re_rank_model=cross_encoder
        )
    )

In [ ]:
os_predictions_df = pd.DataFrame(os_predictions)
print(os_predictions_df.head())

In [ ]:
raw_accuracy = sum(
    [p["retrieved_correct_position"] == 0 for p in os_predictions]
) / len(os_predictions)
reranked_accuracy = sum(
    [p["reranked_correct_position"] == 0 for p in os_predictions]
) / len(os_predictions)

In [ ]:
print(f"Accuracy without re-ranking: {raw_accuracy}")  # 0.5017421602787456
print(f"Accuracy with re-ranking: {reranked_accuracy}")  # 0.6202090592334495

In [ ]:
# do recall @ 1, 3, 5, 10
OPEN_SOURCE_RETRIEVAL = []
OPEN_SOURCE_RETRIEVAL_PLUS_PRE_CE = []
for k in (1, 3, 5, 10, 25, 50):
    embedding_only_recall = os_predictions_df[
        os_predictions_df["retrieved_correct_position"] < k
    ].shape[0]
    reranked_recall = os_predictions_df[
        os_predictions_df["reranked_correct_position"] < k
    ].shape[0]
    print(k, embedding_only_recall, reranked_recall)
    OPEN_SOURCE_RETRIEVAL.append(embedding_only_recall / os_predictions_df.shape[0])
    OPEN_SOURCE_RETRIEVAL_PLUS_PRE_CE.append(
        reranked_recall / os_predictions_df.shape[0]
    )

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
X = [1, 3, 5, 10, 25, 50]

In [ ]:
OPENAI_RETRIEVAL = [
    866 / os_predictions_df.shape[0],
    1009 / os_predictions_df.shape[0],
    1048 / os_predictions_df.shape[0],
    1072 / os_predictions_df.shape[0],
    1108 / os_predictions_df.shape[0],
    1122 / os_predictions_df.shape[0],
]
OLD_CROSS_ENCODER = [
    956 / os_predictions_df.shape[0],
    1057 / os_predictions_df.shape[0],
    1077 / os_predictions_df.shape[0],
    1094 / os_predictions_df.shape[0],
    1120 / os_predictions_df.shape[0],
    1122 / os_predictions_df.shape[0],
]

In [ ]:
# Creating the plot
plt.figure(figsize=(10, 6))
plt.plot(X, OPENAI_RETRIEVAL, label="OAI Retrieval Only", marker="o")
plt.plot(X, OLD_CROSS_ENCODER, label="OAI + Pretrained CE", marker="s")

In [ ]:
plt.plot(X, OPEN_SOURCE_RETRIEVAL, label="OS Retrieval Only", marker="*")
plt.plot(X, OPEN_SOURCE_RETRIEVAL_PLUS_PRE_CE, label="OS + Pretrained CE", marker="^")

In [ ]:
# Adding titles and labels
plt.title(
    "Comparing embedding models + pre-trained vs fine-tuned CE (all retrieved 50 results then re-ranked)"
)
plt.xlabel("Recall @")
plt.ylabel("Performance")
plt.xticks(X)
plt.yticks([i / 100 for i in range(70, 101, 5)])  # Adjusting y-ticks to start from 0.75

In [ ]:
# Adding legend
plt.legend()

In [ ]:
# Show the plot
plt.grid(True)
plt.show()

## Advanced: Fine-tuning the re-ranker

In [ ]:
print(dataset["train"][0])

In [ ]:
from random import shuffle

In [ ]:
shuffled_training_passages = dataset["train"]["question"].copy()
shuffle(shuffled_training_passages)

In [ ]:
unique_train_passages = list(set(dataset["train"]["context"]))
print(len(unique_train_passages), len(dataset["train"]["context"]))  # (9916, 11590)

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
# We will use this embedder to find negative examples
bi_encoder = SentenceTransformer("sentence-transformers/msmarco-MiniLM-L-6-v3")

In [ ]:
embeddings = bi_encoder.encode(
    unique_train_passages,
    show_progress_bar=True,
    batch_size=16,
    normalize_embeddings=True,
)

In [ ]:
print(len(unique_train_passages), embeddings.shape)  # (9916, (9916, 384))

In [ ]:
# use sentence_transformers.util.semantic_search
from sentence_transformers.util import semantic_search

In [ ]:
unique_train_passages = np.array(unique_train_passages)

In [ ]:
# Function to find most similar document
def find_most_similar(text, embeddings, documents, k=3):
    query_embedding = bi_encoder.encode([text], show_progress_bar=False)
    similarities = semantic_search(query_embedding, embeddings, top_k=k)
    return [
        (documents[sim["corpus_id"]], sim["score"], sim["corpus_id"])
        for sim in similarities[0]
    ]

In [ ]:
# Example usage
input_text = unique_train_passages[0]
print(input_text)

In [ ]:
print(find_most_similar(input_text, embeddings, unique_train_passages))